# M2 - Agentic AI - Improving SQL Generation with Reflection

## 1. Introduction
### 1.1. Lab overview

In this lab, you will explore how the reflection pattern can improve an agentic workflow that converts natural language questions to database queries written in SQL. You’ll see how an agent can spot issues in its own outputs, refine them, and improve it’s response before giving a final answer.

### 🎯 1.2 Learning outcome

You will practice applying the reflection pattern to improve an agentic workflow’s ability to write SQL queries. To do this, you will code a reflection step into the workflow, so the agent:

* Reviews its own intermediate results (such as draft SQL or tool outputs).
* Identifies errors or gaps.
* Checks its responses and tool use.
* Refines the output before submitting the final answer.

## 2. Setup: Initialize Environment and Client

In this step, you will prepare your workspace so you can start coding right away. You will:

1. **Import core Python libraries**

   * `json` for handling structured data.
   * `pandas` for working with tabular data.
   * `dotenv` to load environment variables (e.g., API keys).

2. **Load environment variables**
   This ensures your workspace is properly configured with the necessary keys and settings.

3. **Import the `utils.py` module**
   This file contains helper functions that you will use to format outputs and support later steps in the workflow.

**Note:** If you want to explore the contents of `utils.py`, go to the top menu and select **File > Open**.


In [2]:
import json
import utils
import pandas as pd
import aisuite as ai
from dotenv import load_dotenv

load_dotenv()

False

### 2.1 Getting started with AISuite

The labs in this course will make use of a package called aisuite [aisuite repository](https://github.com/andrewyng/aisuite)  that makes it easy to call LLMs hosted by different model providers. Andrew will discuss more details of aisuite in Module 3. Now, initialize the `aisuite client`. This client gives you a single, unified way to connect and interact with different LLMs — so you don’t have to worry about each model having its own setup.

In [3]:
ai_client = ai.Client()

### 2.2. Set Up the Database

In this step, you will create a local SQLite database called **`products.db`**.
The database will be automatically filled with randomly generated product data.

You will use this data later in the lab to practice and test your SQL queries.

In [5]:
utils.create_transactions_db()
utils.print_html(utils.get_schema('products.db'))

SQLite database 'products.db' created with a single 'transactions' table (event-sourced).


In this table, every row represents an **event** (insert, restock, sale, or price update). Analytics such as stock levels, sales, or pricing trends are all derived from these events.

<div style="border:1px solid #93c5fd; border-left:6px solid #3b82f6; background:#eff6ff; border-radius:6px; padding:12px 14px; color:#1e3a8a; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">  
<strong>🔎 Schema overview:</strong><br><br>  
• <code>id</code> → unique event ID (autoincrement).<br>  
• <code>product_id</code>, <code>product_name</code>, <code>brand</code>, <code>category</code>, <code>color</code> → identify the product.<br>  
• <code>action</code> → type of event (<em>insert</em>, <em>restock</em>, <em>sale</em>, <em>price_update</em>).<br>  
• <code>qty_delta</code> → stock change (+ for insert/restock, – for sale, 0 for price updates).<br>  
• <code>unit_price</code> → price at that moment (NULL for restock).<br>  
• <code>notes</code> → optional description of the event.<br>  
• <code>ts</code> → timestamp when the event was logged.<br>  
</div>  

> With this schema, you can always reconstruct the **current state** (stock, latest price, total sales) by aggregating the event history.

## 3. Build a SQL generator

### 3.1. Use an LLM to Query a Database

In this step, you will use a function that turns your natural-language questions into SQL queries.

You provide your question and the database schema as input. The LLM then generates the SQL query that answers your question.

This way, **you** can focus on asking questions while the model takes care of writing the query.


In [ ]:
def generate_sql(question: str, schema: str, model: str) -> str:
    """
    Generate a SQL query from a natural language question and a database schema.

    Args:
        question (str): The natural language question to be answered.
        schema (str): The database schema in a structured format.
        model (str): The name of the LLM model to use for query generation.
    """
    prompt = f"""
    You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

    Schema:
    {schema}

    User question:
    {question}

    Respond with the SQL only.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()